# 개별 미션 4차 — 의료/헬스케어 관련 기술 보증 데이터 분석

- 데이터 : `06_Tech_Credit_Data.xlsx` 의 `기술데이터` 시트 (2,213행 × 17열)
- 목표 변수 : `진행상태` (여신승인 1,808건 / 여신거부 405건)
    - 여신 : 금융기관이 돈을 빌려주는 것 → 여신승인 = 대출 승인
- 실행 환경 : `dur_mission` 폴더의 `.venv` (Python 3.12.10, TensorFlow 2.21 + Keras 3)

In [49]:
# 라이브러리 불러오기와 데이터 확인


import pandas as pd
import numpy as np

df1 = pd.read_excel('data/06_Tech_Credit_Data.xlsx', sheet_name='기술데이터')
print(df1.shape)
df1.head(3)


# → 엑셀 파일에서 '기술데이터' 시트를 읽어 df1에 담겠다
# → 크기가 (2213, 17)인지 확인하고, 앞의 3행으로 컬럼을 보겠다

(2213, 17)


,회사명,기술명,헬스케어 세부분류,기술분류,설립일,신청일,평가기관,평가등급,신청은행,진행상태,대출금액,자본금,설립일자,설립연도,대출금,총합,신청일자
0,Big5시스템,"돼지 태반 추출물을 함유하는 갱년기/폐경기 장애 개선용 식품 조성물, 이를 포함하는...",건강기능식품,바이오 의료,2004/12/23,2011-03-30 00:00:00,기술보증기금,BBB,국민은행,여신승인,50,500,2004-12-23,2004,50,550,2011-03-30
1,박박사LAB항공,준연속식 초임계 추출 기술을 이용한 건강기능식품 소재,건강기능식품,바이오 의료,2000/04/19,2011-11-21 00:00:00,기술보증기금,A,국민은행,여신승인,100,940,2000-04-19,2000,100,1040,2011-11-21
2,TAX네트워크,결명자 주정추출물을 이용한 치매치료용 건강기능식품 및 의약품의 개발,건강기능식품/의약품,바이오 의료,2009/09/09,2011-02-28 00:00:00,기술보증기금,B,기업은행,여신승인,0,100,2009-09-09,2009,0,100,2011-02-28


## 문제 1

> 1. 대출 승인이 된 기업 중, 자본금이 가장 많은 100개 기업에 대하여 추출하고, 해당 기업의 기술명 중 가장 빈도수가 높은 단어를 확인하시오.

- 풀이 흐름
    1. 필터 : 진행상태 == '여신승인'
    2. 상위 100개 : 자본금 기준 nlargest(100)
    3. 기술명 꺼내기 → 명사 추출 (Okt)
    4. 단어 세기 → 가장 많이 나온 단어 확인

In [50]:
# 1번 — 대출 승인된 기업만 필터링


approved = df1[df1['진행상태'] == '여신승인']
print(approved.shape)


# → 진행상태가 '여신승인'인 행만 남겨 approved에 담겠다
#   → 오늘 DUR의 cond1 = (df1['금기유형']=='병용금기')와 같은 방식

(1808, 17)


In [51]:
# 1번 — 자본금 상위 100개 기업 추출


top100 = approved.nlargest(100, '자본금')
print(top100.shape)
top100[['회사명', '자본금', '기술명']].head(10)


# → approved에서 자본금이 가장 큰 100행을 골라 top100에 담겠다
# → 회사명·자본금·기술명만 골라서 상위 10개를 눈으로 확인하겠다

(100, 17)


,회사명,자본금,기술명
970,매너상훈텍티스,9135,인공 와우 정밀 절삭 가공 기술 설계 및 제조
1717,코만도화학,7633,고정밀 투석용 필터 개발
2037,다영텍티스,6864,CT 영상 기반 간질환 멀티모달 융합 진단 모델
1035,경주파트너즈,6653,AI 기반 만성질환 통합관리 플랫폼 상용화 기술
680,미니마니모그룹,6557,디지털 X선 촬영장치용 자동 노출 제어 고도화
757,강산산업,6197,내시경용 자동 노출 제어 상용화 기술
775,박박사LAB산업,6049,비침습 가축 질병 조기경보 시스템 국산화 개발
911,미니마니모솔루션,5878,고정밀 무균 충전 공정 설계 및 제조
42,유진텍티스,5744,치과용 임플란트
1937,클로벳솔루션,5680,파킨슨병 치료제용 리포좀 약물전달 국산화 개발


In [52]:
# 1번 — 형태소 분석기와 명사 추출 함수 준비


from konlpy.tag import Okt

okt = Okt()
stop_list = pd.read_csv('data/stopword.txt')['불용어'].tolist()

def get_nouns(text):
    result = []
    for word in okt.nouns(str(text)):
        if (word not in stop_list) & (len(word) > 1):
            result.append(word)
    return result

print(get_nouns(top100['기술명'].iloc[0]))


# → 한국어 문장에서 명사를 뽑는 Okt 분석기를 준비하겠다
# → 오늘 DUR에서 쓴 기본 불용어 파일(stopword.txt)을 불러오겠다
# → 문장 하나를 받아, 불용어가 아니고 2글자 이상인 명사만 돌려주는 함수를 만들겠다
# → 첫 번째 기술명으로 함수가 잘 동작하는지 시험해 보겠다

['인공', '와우', '정밀', '절삭', '가공', '기술', '설계', '제조']


In [53]:
# 1번 — 기술명 명사 빈도 확인


from collections import Counter

top100_nouns = top100['기술명'].apply(get_nouns)
word_count = Counter(word for nouns in top100_nouns for word in nouns)
word_count.most_common(20)


# → 기술명 100개를 문장마다 따로 명사 리스트로 바꾸겠다 (.apply = 한 문장씩 처리)
# → 모든 명사 리스트를 하나로 이어서, 단어마다 몇 번 나왔는지 세겠다
# → 가장 많이 나온 단어 20개를 (단어, 횟수)로 보겠다

[('개발', 36),
 ('기술', 30),
 ('기능', 21),
 ('제조', 20),
 ('기반', 20),
 ('건강', 20),
 ('치료', 19),
 ('이용', 18),
 ('설계', 14),
 ('고도화', 13),
 ('식품', 13),
 ('소재', 13),
 ('적용', 13),
 ('진단', 12),
 ('플랫폼', 12),
 ('영상', 11),
 ('정밀', 9),
 ('상용', 9),
 ('양산', 8),
 ('기기', 8)]

In [54]:
# 1번 — 일반 기술 용어를 제외한 빈도 확인 (추가 분석)


tech_stop = ['개발', '기술', '기능', '제조', '기반', '이용', '설계', '고도화', '적용', '상용', '양산']

word_count2 = Counter({word: cnt for word, cnt in word_count.items() if word not in tech_stop})
word_count2.most_common(10)


# → 어떤 기술명에나 붙는 일반 용어 11개를 기술 불용어(tech_stop)로 정하겠다
# → 앞에서 센 결과(word_count)에서 이 단어들만 빼고 새로 담겠다
#   → {단어: 횟수 for ...}: 조건에 맞는 것만 골라 사전을 새로 만드는 방식 (사전 컴프리헨션)
# → 남은 단어 중 가장 많이 나온 10개를 보겠다

[('건강', 20),
 ('치료', 19),
 ('식품', 13),
 ('소재', 13),
 ('진단', 12),
 ('플랫폼', 12),
 ('영상', 11),
 ('정밀', 9),
 ('기기', 8),
 ('휴대', 7)]

### 문제 1 결론

> 1. 대출 승인이 된 기업 중, 자본금이 가장 많은 100개 기업에 대하여 추출하고, 해당 기업의 기술명 중 가장 빈도수가 높은 단어를 확인하시오.

- 자본금 상위 100개 승인 기업의 기술명에서 **가장 빈도수가 높은 단어는 '개발'(36회)**
- 다만 상위 단어(개발 · 기술 · 제조 · 기반 · 설계 등)는 어떤 기술명에나 붙는 일반 용어라, 기업의 기술 분야를 설명하지 못함
- 일반 용어를 제외하면 '건강'(20), '치료'(19), '식품'(13), '소재'(13), '진단'(12), '플랫폼'(12) 등이 상위 → 건강기능식품 · 치료 · 진단 분야 기업이 많음
- 참고 : 상위 100행 중 같은 회사명이 두 번 나온 곳이 4곳(실제 회사명 96개)
    - 같은 회사명이어도 행마다 자본금이 달라, 회사명을 기업 구분 기준으로 쓰기 어려움 → 신청 1건 = 기업 1개로 보고 100행 사용
- 한계 : Okt가 '건강기능식품' 같은 합성어를 '건강 / 기능 / 식품'으로 나누어 세므로, 합성어 빈도가 분산되어 나타남

## 문제 2

> 2. 기업의 설립일을 이용해, 기업의 업력 (설립 후 현재까지 지속된 기간 / 연도)을 계산하고, 대출 승인여부에 따른 기업 업력의 대표값의 차이가 있는지 검정하시오.

### 문제 조각내기

| 조각 | 컬럼 | 할 일 |
|---|---|---|
| 설립일을 이용해 업력을 계산 | 설립일자 | 현재 연도 − 설립 연도 |
| 대출 승인여부에 따른 | 진행상태 | 승인 / 거부 두 그룹으로 나누기 |
| 업력의 대표값의 차이 | — | 두 그룹의 평균 또는 중앙값 비교 |
| 검정하시오 | — | 두 그룹 비교 검정 (t-검정 또는 Mann-Whitney U) |

- 대표값 : 한 그룹을 숫자 하나로 요약한 값 (평균, 중앙값 등)
- 검정 : 두 그룹의 차이가 우연인지 진짜 차이인지를 p-value로 판단하는 방법

### 먼저 정한 것

1. "현재"의 기준 → **2026년(오늘 연도)**
    - 문제에 "현재까지"라고 되어 있어 오늘 연도 기준으로 계산
    - 대출 신청은 2011 ~ 2013년 → "신청 당시 업력"을 보려면 신청 연도 기준이 더 적절할 수 있음
    - 신청 연도의 차이가 최대 2년이라, 기준을 바꿔도 검정 결과는 크게 달라지지 않을 것으로 예상
2. 설립 연도 1900년(20건) → **결측(NaN) 처리 후 검정에서 제외**
    - 1900-01-01은 설립일을 모를 때 넣은 값으로 판단
    - 그대로 두면 업력이 126년이 되어 평균을 크게 왜곡

### 풀이 흐름

1. 업력 = 2026 − 설립 연도
2. 설립 연도 1900년은 결측 처리
3. 승인 / 거부 그룹별 업력 요약 (평균, 중앙값, 분포)
4. 정규성 확인 → 알맞은 검정 선택 → 대표값 차이 검정

In [55]:
# 2번 — 업력 계산 (현재 연도 기준, 1900년은 결측 처리)


df1['설립일자'] = pd.to_datetime(df1['설립일자'])
df1['업력'] = 2026 - df1['설립일자'].dt.year
df1.loc[df1['설립일자'].dt.year == 1900, '업력'] = np.nan

print('1900년 결측 처리 :', df1['업력'].isna().sum(), '건')
df1['업력'].describe()


# → 설립일자 글자('2004-12-23')를 날짜 자료형으로 바꾸겠다 → .dt.year로 연도만 꺼낼 수 있게
# → 2026에서 설립 연도를 빼서 업력(년) 컬럼을 새로 만들겠다
# → 설립 연도가 1900인 행은 업력을 NaN(결측)으로 바꾸겠다
#   → df1.loc[조건, '컬럼'] = 값 : 조건에 맞는 행의 그 컬럼만 골라서 값을 바꾸는 방식
# → 결측 처리한 건수(20건)와 업력의 분포(평균, 최소, 최대 등)를 확인하겠다

1900년 결측 처리 : 20 건


count    2193.000000
mean       20.471956
std         5.458726
min        13.000000
25%        16.000000
50%        19.000000
75%        24.000000
max        44.000000
Name: 업력, dtype: float64

In [56]:
# 2번 — 승인 여부별 업력 요약


df1.groupby('진행상태')['업력'].describe()


# → 진행상태(여신승인 / 여신거부)별로 나누어 업력의 개수·평균·중앙값(50%)·최소·최대를 한 번에 보겠다

,count,mean,std,min,25%,50%,75%,max
진행상태,,,,,,,,
여신거부,401.0,20.493766,5.621885,13.0,16.0,19.0,24.0,40.0
여신승인,1792.0,20.467076,5.423137,13.0,16.0,19.0,24.0,44.0


In [57]:
# 2번 — 그룹별 정규성 확인


from scipy import stats

age_df = df1.dropna(subset=['업력'])
age_yes = age_df.loc[age_df['진행상태'] == '여신승인', '업력']
age_no = age_df.loc[age_df['진행상태'] == '여신거부', '업력']

print('승인 그룹 p-value :', stats.shapiro(age_yes).pvalue)
print('거부 그룹 p-value :', stats.shapiro(age_no).pvalue)
print('왜도 (승인 / 거부) :', round(age_yes.skew(), 3), '/', round(age_no.skew(), 3))


# → 업력이 결측인 20건을 빼고 age_df에 담겠다
# → 승인 그룹과 거부 그룹의 업력을 각각 꺼내겠다
# → Shapiro-Wilk 검정으로 두 그룹이 정규분포를 따르는지 확인하겠다
#   → 귀무가설 : 정규분포를 따른다 → p-value < 0.05면 정규분포가 아니라고 판단
# → 왜도(skew)로 분포가 어느 쪽으로 얼마나 치우쳤는지 숫자로 보겠다 (0이면 대칭, 양수면 오른쪽 꼬리)

승인 그룹 p-value : 2.6485962009542807e-30
거부 그룹 p-value : 8.314234733008606e-15
왜도 (승인 / 거부) : 0.988 / 1.012


In [58]:
# 2번 — 승인 여부에 따른 업력 차이 검정


mw = stats.mannwhitneyu(age_yes, age_no, alternative='two-sided')
print('Mann-Whitney U (주 검정) p-value :', round(mw.pvalue, 4))

lev = stats.levene(age_yes, age_no)
welch = stats.ttest_ind(age_yes, age_no, equal_var=False)
print('Levene 등분산 p-value :', round(lev.pvalue, 4))
print('Welch t-검정 (참고) p-value :', round(welch.pvalue, 4))


# → 정규분포가 아니므로, 순위로 비교하는 Mann-Whitney U 검정을 주 검정으로 하겠다
#   → alternative='two-sided' : "승인이 더 크다/작다"가 아니라 "다르다"를 검정하겠다 (양측 검정)
# → 참고로 평균을 비교하는 t-검정도 함께 보겠다
#   → Levene 검정 : 두 그룹의 퍼진 정도(분산)가 같은지 확인
#   → equal_var=False (Welch) : 분산이 같다고 가정하지 않는 안전한 t-검정
#   → 표본이 각각 400개, 1,800개 정도로 많아 정규성이 깨져도 t-검정 결과를 참고용으로 볼 수 있다

Mann-Whitney U (주 검정) p-value : 0.8405
Levene 등분산 p-value : 0.574
Welch t-검정 (참고) p-value : 0.9311


### 문제 2 결론

> 2. 기업의 설립일을 이용해, 기업의 업력 (설립 후 현재까지 지속된 기간 / 연도)을 계산하고, 대출 승인여부에 따른 기업 업력의 대표값의 차이가 있는지 검정하시오.

- 업력 요약 : 승인 그룹 평균 20.47년 · 중앙값 19년 / 거부 그룹 평균 20.49년 · 중앙값 19년
- 정규성 : 두 그룹 모두 Shapiro-Wilk p-value < 0.001, 왜도 약 1 → 정규분포가 아닌 오른쪽 꼬리 분포
- 검정 선택 : 정규성이 깨졌으므로 비모수 검정인 **Mann-Whitney U 검정**을 주 검정으로 사용
    - 귀무가설 : 승인 그룹과 거부 그룹의 업력 분포(대표값)에 차이가 없다
- 결과 : Mann-Whitney U p-value = 0.840 > 0.05 → **귀무가설을 기각하지 못함, 대출 승인 여부에 따른 업력 차이는 없음**
    - 참고 : Levene p-value = 0.574로 두 그룹의 분산 차이도 없고, Welch t-검정 p-value = 0.931로 같은 결론
- 해석 : 업력은 대출 승인 여부를 가르는 요인으로 보기 어려움
- 참고 : 업력 기준을 "현재(2026년)" 대신 "신청 연도"로 바꿔도 Mann-Whitney U p-value 약 0.855로 결론이 같음
- 한계 : 설립 연도 1900년(20건)은 설립일을 모르는 값으로 보고 제외함

## 문제 3

> 3. 기업 평가등급과 대출 승인여부의 유의미한 차이가 있는지 가설 검정을 수행하고, 대출 승인 비율을 각 평가 등급 별로 계산하시오.

### 문제 조각내기

| 조각 | 컬럼 | 할 일 |
|---|---|---|
| 기업 평가등급과 대출 승인여부의 | 평가등급, 진행상태 | 두 범주형 변수의 교차표 만들기 |
| 유의미한 차이가 있는지 가설 검정 | — | 카이제곱 독립성 검정 |
| 대출 승인 비율을 각 평가 등급 별로 | 평가등급, 진행상태 | 등급별 승인 건수 ÷ 등급별 전체 건수 |

- 범주형 변수 : 숫자 크기가 아니라 종류(등급, 은행 이름 등)를 나타내는 변수
- 카이제곱 독립성 검정 : 두 범주형 변수가 서로 관련이 있는지(독립인지) 확인하는 검정
    - 귀무가설 : 평가등급과 승인 여부는 관련이 없다(독립이다)

### 등급 정리 규칙

| 규칙 | 대상 | 처리 | 이유 |
|---|---|---|---|
| ① | `T3/A`, `T8/CC` 등 | `/` 뒤의 신용등급만 사용 | 나머지 등급과 같은 기준으로 비교하기 위해 |
| ② | T1, T2 (3건) | 제외 | 신용등급 정보가 없어 다른 등급과 맞출 수 없음 |
| ③ | 건수가 아주 적은 끝 등급 | AAA + AA → AA 이상, CCC + CC + C → CCC 이하 | 카이제곱 기대빈도 조건을 맞추기 위해 인접 등급끼리 병합 |

- 기대빈도 조건 : 기대빈도 5 미만 칸이 전체의 20% 이하, 1 미만 칸은 없어야 검정 결과를 믿을 수 있음

In [59]:
# 3번 — 평가등급별 승인 여부 교차표와 승인 비율 (원래 등급)


grade_ct = pd.crosstab(df1['평가등급'], df1['진행상태'])
grade_ct['합계'] = grade_ct['여신거부'] + grade_ct['여신승인']
grade_ct['승인비율'] = (grade_ct['여신승인'] / grade_ct['합계']).round(3)
grade_ct.sort_values('합계', ascending=False)


# → 평가등급(행) × 진행상태(열)로 건수를 세는 교차표를 만들겠다
# → 등급별 전체 건수(합계)와 승인 비율(승인 ÷ 합계)을 컬럼으로 추가하겠다
# → 건수가 많은 등급부터 보이게 정렬하겠다

진행상태,여신거부,여신승인,합계,승인비율
평가등급,,,,
B,122,474,596,0.795
BB,110,483,593,0.815
BBB,95,480,575,0.835
A,57,255,312,0.817
AA,11,76,87,0.874
T3/A,3,15,18,0.833
T2/AA,1,9,10,0.900
T4/BBB,3,5,8,0.625
T8/CC,2,1,3,0.333


In [60]:
# 3번 — 평가등급 정리 (신용등급 기준으로 통일, 희소 등급 병합)


grade_map = {'AAA':'AA 이상', 'AA':'AA 이상', 'A':'A', 'BBB':'BBB', 'BB':'BB', 'B':'B',
             'CCC':'CCC 이하', 'CC':'CCC 이하', 'C':'CCC 이하'}
grade_order = ['AA 이상', 'A', 'BBB', 'BB', 'B', 'CCC 이하']

df1['등급정리'] = df1['평가등급'].str.split('/').str[-1].map(grade_map)
print('제외된 건수 (T1, T2) :', df1['등급정리'].isna().sum())

grade_ct2 = pd.crosstab(df1['등급정리'], df1['진행상태']).reindex(grade_order)
grade_ct2['합계'] = grade_ct2['여신거부'] + grade_ct2['여신승인']
grade_ct2['승인비율'] = (grade_ct2['여신승인'] / grade_ct2['합계']).round(3)
grade_ct2


# → 원래 등급을 정리된 등급으로 바꾸는 대응표(grade_map)와, 표에 보여줄 등급 순서(grade_order)를 정하겠다
# → '/'로 나눈 뒤 마지막 조각(신용등급)만 꺼내고, 대응표로 바꿔 '등급정리' 컬럼을 만들겠다
#   → 'T3/A' → ['T3', 'A'] → 'A' / 'BB' → ['BB'] → 'BB' / 'T1' → 대응표에 없어서 NaN(제외)
# → 정리된 등급으로 교차표와 승인 비율을 다시 계산하고, 신용등급 순서대로 정렬하겠다

제외된 건수 (T1, T2) : 3


진행상태,여신거부,여신승인,합계,승인비율
등급정리,,,,
AA 이상,12,87,99,0.879
A,60,270,330,0.818
BBB,98,485,583,0.832
BB,110,483,593,0.815
B,122,476,598,0.796
CCC 이하,3,4,7,0.571


In [61]:
# 3번 — 카이제곱 독립성 검정


chi2, p, dof, expected = stats.chi2_contingency(grade_ct2[['여신거부', '여신승인']])
print('카이제곱 통계량 :', round(chi2, 3), '/ 자유도 :', dof, '/ p-value :', round(p, 4))
print('기대빈도 5 미만 칸 :', (expected < 5).sum(), '/', expected.size, '칸')
print('가장 작은 기대빈도 :', round(expected.min(), 2))

chi2_raw, p_raw, _, exp_raw = stats.chi2_contingency(pd.crosstab(df1['평가등급'], df1['진행상태']))
print('(참고) 정리 전 16개 등급 p-value :', round(p_raw, 4), '/ 5 미만 칸 비율 :', round((exp_raw < 5).mean(), 2))


# → 정리된 교차표(건수 두 열만)로 카이제곱 독립성 검정을 하겠다
#   → chi2 : 실제 건수와 기대빈도의 차이를 모은 값 / dof : 자유도 / expected : 칸별 기대빈도
# → 기대빈도 조건(5 미만 칸 20% 이하, 1 미만 칸 없음)을 만족하는지 확인하겠다
# → 참고로 정리 전 원래 등급 그대로 검정한 결과도 함께 보겠다 → 조건을 벗어나 믿기 어려운 결과

카이제곱 통계량 : 8.002 / 자유도 : 5 / p-value : 0.1561
기대빈도 5 미만 칸 : 1 / 12 칸
가장 작은 기대빈도 : 1.28
(참고) 정리 전 16개 등급 p-value : 0.221 / 5 미만 칸 비율 : 0.59


### 문제 3 결론

> 3. 기업 평가등급과 대출 승인여부의 유의미한 차이가 있는지 가설 검정을 수행하고, 대출 승인 비율을 각 평가 등급 별로 계산하시오.

- 등급별 승인 비율 (정리 후) : AA 이상 0.879 / A 0.818 / BBB 0.832 / BB 0.815 / B 0.796 / CCC 이하 0.571
    - 주요 등급(AA 이상 ~ B)의 승인 비율은 약 0.80 ~ 0.88로 비슷하고, 등급이 높을수록 약간 높아지는 경향
    - CCC 이하는 7건뿐이라 비율(0.571)을 일반화하기 어려움
- 카이제곱 독립성 검정 : 카이제곱 = 8.002, 자유도 = 5, **p-value = 0.156 > 0.05**
    - 기대빈도 5 미만 칸 1 / 12칸(약 8%), 최소 기대빈도 1.28 → 조건 만족
    - **귀무가설을 기각하지 못함 → 평가등급과 대출 승인 여부 사이에 통계적으로 유의미한 관련이 없음**
- 참고 : 원래 16개 등급 그대로 검정하면 p-value = 0.221로 결론은 같지만, 기대빈도 5 미만 칸이 약 59%라 결과를 믿기 어려움
- 한계 : 등급 병합 방식에 따라 결과가 달라질 수 있음, 기술등급(T1 ~ T8)은 신용등급과 다른 체계라 뒤쪽 신용등급만 사용함

## 문제 4

> 4. 기술 세분류(헬스케어 세부분류)에 '병원/의료 IT'에 해당하는 기업의 기술명을 단어표현을 통해, '감염'라는 단어와 가장 유사한 단어들이 무엇이 있는지 확인하시오.

### 문제 조각내기

| 조각 | 컬럼 | 할 일 |
|---|---|---|
| '병원/의료 IT'에 해당하는 기업의 | 헬스케어 세부분류 | 필터 |
| 기술명을 | 기술명 | 문장별 명사 리스트로 바꾸기 |
| 단어표현을 통해 | — | Word2Vec으로 단어를 벡터(숫자 목록)로 학습 |
| '감염'과 가장 유사한 단어 | — | most_similar('감염') |

- 단어표현(임베딩) : 단어를 숫자 목록(벡터)으로 바꿔, 함께 쓰이는 단어끼리 가까운 숫자가 되게 만드는 것
- Word2Vec : 주변 단어로 가운데 단어를 맞히는(또는 그 반대) 방식으로 단어 벡터를 학습하는 모델
- 유사도 : 두 단어 벡터의 방향이 얼마나 비슷한지 (코사인 유사도, 1에 가까울수록 비슷)

### 주의할 점

- '병원/의료 IT' 기술명은 183문장뿐이라, Word2Vec 결과가 시드(난수 시작점)에 따라 흔들릴 수 있음
    - 시드 10개로 각각 학습한 뒤 유사도를 평균 내서 안정적인 결과를 보기
- Word2Vec 결과가 실제 문장에서도 맞는지, '감염'과 같은 문장에 함께 나온 단어(동시출현)로 교차 확인

In [62]:
# 4번 — '병원/의료 IT' 기술명 추출과 명사 리스트 만들기


it_text = df1.loc[df1['헬스케어 세부분류'] == '병원/의료 IT', '기술명']
it_nouns = it_text.apply(get_nouns).tolist()

print('문장 수 :', len(it_nouns))
print('예시 :', it_nouns[:3])
print("'감염'이 들어간 문장 수 :", sum('감염' in nouns for nouns in it_nouns))


# → 헬스케어 세부분류가 '병원/의료 IT'인 기업의 기술명만 꺼내겠다 → 183문장
# → 문장마다 명사 리스트로 바꾸고, Word2Vec이 받는 형태(리스트의 리스트)로 만들겠다
# → '감염'이 몇 문장에 나오는지 확인하겠다 → 적으면 결과 해석에 주의

문장 수 : 183
예시 : [[], ['의료', '기관', '전용', '고객', '관리', '솔루션'], []]
'감염'이 들어간 문장 수 : 9


In [63]:
# 4번 — Word2Vec 학습 (시드 10개 평균)


from gensim.models import Word2Vec

sim_list = []
for seed in range(10):
    w2v = Word2Vec(sentences=it_nouns, vector_size=100, window=3, min_count=1, sg=1,
                   epochs=100, workers=1, seed=seed)
    sim_list.append(pd.Series(dict(w2v.wv.most_similar('감염', topn=len(w2v.wv) - 1))))

sim_avg = pd.concat(sim_list, axis=1).mean(axis=1).sort_values(ascending=False)
sim_avg.head(10).round(3)


# → 시드를 0 ~ 9로 바꿔 가며 Word2Vec을 10번 학습하겠다
#   → vector_size=100 : 단어 하나를 숫자 100개로 표현
#   → window=3 : 앞뒤 3단어까지를 "주변 단어"로 봄
#   → min_count=1 : 한 번만 나온 단어도 학습 (문장이 적어서)
#   → sg=1 : Skip-gram 방식 (가운데 단어로 주변 단어 맞히기, 적은 데이터에 유리)
#   → epochs=100 : 문장이 적어서 100번 반복 학습
#   → workers=1, seed=seed : 결과가 매번 같게 나오도록 고정
# → 매번 '감염'과 나머지 모든 단어의 유사도를 구해 모으고, 10번의 평균으로 순위를 매기겠다

모니터링    0.897
관리      0.875
병원      0.815
원가      0.811
스마트     0.789
이력      0.783
병상      0.775
솔루션     0.775
블록      0.772
고객      0.772
dtype: float64

In [64]:
# 4번 — 교차 확인 : '감염'과 같은 문장에 나온 단어 (동시출현)


co_count = Counter(word for nouns in it_nouns if '감염' in nouns for word in nouns if word != '감염')
co_count.most_common(10)


# → '감염'이 들어간 문장만 골라, 그 문장에 함께 나온 단어를 세겠다
# → Word2Vec 상위 단어가 실제로 '감염'과 같이 쓰인 단어인지 비교하겠다

[('적용', 12),
 ('관리', 10),
 ('모니터링', 9),
 ('시스템', 9),
 ('개발', 6),
 ('기반', 2),
 ('플랫폼', 2),
 ('기술', 2),
 ('연동', 2),
 ('실시간', 1)]

### 문제 4 결론

> 4. 기술 세분류(헬스케어 세부분류)에 '병원/의료 IT'에 해당하는 기업의 기술명을 단어표현을 통해, '감염'라는 단어와 가장 유사한 단어들이 무엇이 있는지 확인하시오.

- '병원/의료 IT' 기술명 183문장으로 Word2Vec(Skip-gram, 시드 10개 평균)을 학습
- **'감염'과 가장 유사한 단어 : 모니터링(0.897), 관리(0.875), 병원(0.815), 원가(0.811), 스마트(0.789)**
- 교차 확인 : '감염'은 9문장에 나오며, 모두 "감염관리 모니터링 시스템" 형태 → 동시출현 상위 단어 : 적용(12), 관리(10), 모니터링(9), 시스템(9), 개발(6)
- 해석 : Word2Vec 상위 단어가 '감염관리 모니터링 시스템'을 이루는 단어들과 겹침 → 이 데이터에서 '감염'은 **병원 감염관리 · 모니터링 시스템 기술**의 맥락으로 쓰임
- 한계
    - 문장 수(183개)와 '감염' 등장 문장(9개)이 매우 적어, 의미적 유사성보다 "같은 문장에 자주 붙어 나온 단어"가 주로 잡힘
    - 기술명이 "~를 적용한 ~ 시스템 개발" 같은 틀에 맞춰 만들어진 문장이 많아, 일반 용어(적용 · 시스템 · 개발)의 영향이 큼

## 문제 5

> 5. 대출금액을 가장 많이 집행한 은행을 확인하고, 해당 은행이 어떤 기술 세분류(헬스케어 세부분류)에 대출금액을 가장 많이 집행했는지 계산하시오.

### 문제 조각내기

| 조각 | 컬럼 | 할 일 |
|---|---|---|
| 대출금액을 가장 많이 집행한 은행 | 신청은행, 대출금 | 은행별 대출금 합계 → 1등 은행 |
| 해당 은행이 | 신청은행 | 1등 은행으로 필터 |
| 어떤 기술 세분류에 가장 많이 집행 | 헬스케어 세부분류, 대출금 | 세부분류별 대출금 합계 → 1등 분류 |

### 먼저 정한 것

- `대출금액` 컬럼에는 숫자 사이에 **'금액없음'** 글자가 9건 섞여 있어 바로 합계를 낼 수 없음
- 같은 값을 숫자로 정리해 둔 **`대출금`** 컬럼을 사용 ('금액없음' → 0)
- "집행"은 실제로 돈이 나간 것이므로, 대출금 합계 = 집행 금액으로 봄 (거부된 건은 대부분 0원)

In [65]:
# 5번 — 두 대출 컬럼 비교 (대출금 컬럼을 쓰는 근거)


print(df1['대출금액'].apply(type).value_counts())
print(df1.loc[pd.to_numeric(df1['대출금액'], errors='coerce').isna(), ['대출금액', '대출금']].drop_duplicates())
print('숫자인 행에서 두 컬럼이 다른 건수 :',
      (pd.to_numeric(df1['대출금액'], errors='coerce').dropna() != df1['대출금'][pd.to_numeric(df1['대출금액'], errors='coerce').notna()]).sum())


# → 대출금액 컬럼에 숫자와 글자가 몇 개씩 섞여 있는지 보겠다
# → 숫자로 바꿀 수 없는 값('금액없음')이 대출금 컬럼에서는 무엇으로 되어 있는지 보겠다 → 0
#   → pd.to_numeric(errors='coerce') : 숫자로 못 바꾸는 값은 NaN으로 바꾸는 옵션
# → 나머지 행은 두 컬럼 값이 같은지 확인하겠다 → 0건이면 대출금 = 대출금액을 숫자로 정리한 컬럼

대출금액
<class 'int'>    2204
<class 'str'>       9
Name: count, dtype: int64
     대출금액  대출금
493  금액없음    0
숫자인 행에서 두 컬럼이 다른 건수 : 0


In [66]:
# 5번 — 은행별 대출금 합계


bank_sum = df1.groupby('신청은행')['대출금'].sum().sort_values(ascending=False)
top_bank = bank_sum.index[0]
print('대출금을 가장 많이 집행한 은행 :', top_bank)
bank_sum.head(5)


# → 은행별로 묶어 대출금을 모두 더하고, 큰 순서로 정렬하겠다
# → 1등 은행 이름을 top_bank에 담아 다음 셀에서 쓰겠다

대출금을 가장 많이 집행한 은행 : 기업은행


신청은행
기업은행    84285
국민은행    26524
우리은행    25425
신한은행    13572
대구은행     2890
Name: 대출금, dtype: int64

In [67]:
# 5번 — 1등 은행의 헬스케어 세부분류별 대출금 합계


bank_df = df1[df1['신청은행'] == top_bank]
field_sum = bank_df.groupby('헬스케어 세부분류')['대출금'].agg(['sum', 'count']).sort_values('sum', ascending=False)
field_sum['비중(%)'] = (field_sum['sum'] / field_sum['sum'].sum() * 100).round(1)
field_sum.head(10)


# → 1등 은행의 신청 건만 남기겠다
# → 헬스케어 세부분류별로 대출금 합계(sum)와 건수(count)를 함께 구하고, 합계 순으로 정렬하겠다
#   → .agg(['sum', 'count']) : 한 번에 여러 가지 계산을 하는 방법
# → 은행 전체 대출금 중 각 분류가 차지하는 비중(%)도 계산하겠다

,sum,count,비중(%)
헬스케어 세부분류,,,
의료영상 AI,7704,145,9.1
건강기능식품,7370,118,8.7
의료영상 장치,7310,76,8.7
병원/의료 IT,6470,100,7.7
수술/중재기기,6100,45,7.2
의료기기(치과),5271,5,6.3
신약/바이오의약품,4690,97,5.6
제형/약물전달,3620,59,4.3
의료기기 부품/제조,3520,26,4.2


### 문제 5 결론

> 5. 대출금액을 가장 많이 집행한 은행을 확인하고, 해당 은행이 어떤 기술 세분류(헬스케어 세부분류)에 대출금액을 가장 많이 집행했는지 계산하시오.

- 대출금을 가장 많이 집행한 은행 : **기업은행** (합계 84,285) → 2위 국민은행(26,524)의 약 3배
- 기업은행이 가장 많이 집행한 헬스케어 세부분류 : **의료영상 AI (7,704, 기업은행 전체의 9.1%)**
    - 상위 분류 : 의료영상 AI 7,704 → 건강기능식품 7,370 → 의료영상 장치 7,310 → 병원/의료 IT 6,470
    - 1위와 2 ~ 3위의 차이가 크지 않아, 특정 분야에 몰렸다기보다 의료영상 · 건강기능식품 · 병원/의료 IT 등에 고르게 분산
- 참고 : `대출금액` 컬럼의 '금액없음' 9건은 `대출금` 컬럼에서 0으로 되어 있어, 숫자 컬럼인 `대출금`을 사용
- 한계 : 금액 단위가 명시되어 있지 않아(백만 원 단위로 추정) 절대 금액 해석은 주의

## 문제 6

> 6. 기술명과 대출금액, 기술분류, 신청은행, 자본금을 입력했을 때, 대출 승인 여부를 판별하는 모델을 Scikit Learn라이브러리를 활용해 구성하시오.

### 문제 조각내기

| 입력 변수 | 종류 | 전처리 |
|---|---|---|
| 기술명 | 텍스트 | 명사 추출 → TF-IDF (단어 중요도 숫자로 바꾸기) |
| 대출금액 (대출금), 자본금 | 숫자 | 표준화 (StandardScaler) |
| 기술분류, 신청은행 | 범주형 | 원-핫 인코딩 (OneHotEncoder) |

- Pipeline : 전처리와 모델을 한 줄로 이어, 학습 · 예측 때 같은 순서로 자동 적용하는 도구
- ColumnTransformer : 컬럼마다 다른 전처리를 한 번에 적용하는 도구
- TF-IDF : 한 문장에 자주 나오지만 전체 문장에는 드물게 나오는 단어일수록 큰 값을 주는 방식
- 원-핫 인코딩 : 범주 하나를 "해당하면 1, 아니면 0"인 컬럼 여러 개로 바꾸는 방식
- 모델 : 로지스틱 회귀 (class_weight='balanced'로 불균형 보정)
    - RandomForest는 이 PC에서 스마트 앱 컨트롤이 sklearn.ensemble을 차단한 적이 있어 로지스틱 회귀 사용

### 주의 : 대출금액의 타깃 누수

- 대출금액은 **승인 후에 정해지는 값**이라, 거부된 405건 중 404건이 0원
- 입력에 넣으면 모델이 "0원이면 거부"를 그대로 베껴 성능이 비정상적으로 높게 나옴 → **타깃 누수(target leakage)**
- 문제의 요구대로 넣은 모델(A)과, 대출금을 뺀 모델(B)을 함께 만들어 비교

In [68]:
# 6번 — 입력(X) · 정답(Y) 준비와 학습 / 평가 분리


from sklearn.model_selection import train_test_split

X6 = df1[['기술명', '대출금', '기술분류', '신청은행', '자본금']]
Y6 = (df1['진행상태'] == '여신승인').astype(int)

X6_train, X6_test, Y6_train, Y6_test = train_test_split(X6, Y6, test_size=0.25, random_state=1234, stratify=Y6)
print(X6_train.shape, X6_test.shape)
print(Y6_train.value_counts(normalize=True).round(3))


# → 문제에서 지정한 5개 컬럼을 입력으로 쓰겠다 (대출금액은 숫자로 정리된 대출금 컬럼 사용)
# → 여신승인이면 1, 여신거부면 0으로 정답을 만들겠다
# → 학습 75% / 평가 25%로 나누되, 승인 비율(약 82:18)을 양쪽에 똑같이 유지하겠다 (stratify)

(1659, 5) (554, 5)
진행상태
1    0.817
0    0.183
Name: proportion, dtype: float64


In [69]:
# 6번 — 전처리 + 모델 파이프라인 정의


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

def make_model(num_cols):
    preprocess = ColumnTransformer([
        ('text', TfidfVectorizer(tokenizer=get_nouns, token_pattern=None, lowercase=False), '기술명'),
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['기술분류', '신청은행']),
    ])
    return Pipeline([
        ('preprocess', preprocess),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000)),
    ])


# → 전처리와 모델을 묶은 파이프라인을 만드는 함수를 정의하겠다 (숫자 컬럼만 바꿔 두 모델을 만들기 위해)
#   → text : 기술명을 명사(get_nouns)로 쪼갠 뒤 TF-IDF 숫자로 바꾸겠다
#     → token_pattern=None : 기본 쪼개기 규칙 대신 get_nouns만 쓰겠다는 표시
#   → num : 숫자 컬럼을 평균 0, 표준편차 1로 맞추겠다 (자본금처럼 큰 숫자가 결과를 좌우하지 않게)
#   → cat : 기술분류와 신청은행을 원-핫 인코딩하겠다
#     → handle_unknown='ignore' : 평가 데이터에만 있는 은행이 나와도 에러 없이 0으로 처리
#   → model : 로지스틱 회귀, class_weight='balanced'로 적은 쪽(거부)에 더 큰 가중치

In [70]:
# 6번 — 모델 A (문제 요구대로 5개 컬럼 모두 사용)


model_A = make_model(['대출금', '자본금'])
model_A.fit(X6_train, Y6_train)
pred_A = model_A.predict(X6_test)
print(classification_report(Y6_test, pred_A, target_names=['여신거부', '여신승인'], digits=3))


# → 대출금 · 자본금을 숫자 입력으로 넣은 모델을 학습하겠다
# → 평가 데이터로 예측하고, 클래스별 precision · recall · F1을 보겠다

              precision    recall  f1-score   support

        여신거부      0.856     1.000     0.922       101
        여신승인      1.000     0.962     0.981       453

    accuracy                          0.969       554
   macro avg      0.928     0.981     0.952       554
weighted avg      0.974     0.969     0.970       554



In [71]:
# 6번 — 모델 B (누수 확인용 : 대출금 제외)


model_B = make_model(['자본금'])
model_B.fit(X6_train, Y6_train)
pred_B = model_B.predict(X6_test)
print(classification_report(Y6_test, pred_B, target_names=['여신거부', '여신승인'], digits=3))

print('macro F1 비교 — A :', round(f1_score(Y6_test, pred_A, average='macro'), 3),
      '/ B :', round(f1_score(Y6_test, pred_B, average='macro'), 3))


# → 대출금만 빼고 나머지는 똑같은 모델을 학습하겠다
# → 두 모델의 macro F1(승인 · 거부 F1의 단순 평균)을 비교하겠다
#   → 차이가 크면 모델 A의 성능은 대출금(승인 후 정해지는 값) 덕분 = 타깃 누수

              precision    recall  f1-score   support

        여신거부      0.182     0.376     0.245       101
        여신승인      0.817     0.623     0.707       453

    accuracy                          0.578       554
   macro avg      0.500     0.499     0.476       554
weighted avg      0.702     0.578     0.623       554

macro F1 비교 — A : 0.952 / B : 0.476


### 문제 6 결론

> 6. 기술명과 대출금액, 기술분류, 신청은행, 자본금을 입력했을 때, 대출 승인 여부를 판별하는 모델을 Scikit Learn라이브러리를 활용해 구성하시오.

- 구성 : Pipeline + ColumnTransformer (기술명 → 명사 TF-IDF / 대출금 · 자본금 → 표준화 / 기술분류 · 신청은행 → 원-핫) + 로지스틱 회귀(class_weight='balanced')
- 모델 A (문제 요구 5개 컬럼) : 평가 데이터 macro F1 = **0.952**, 여신거부 recall = 1.000
- 모델 B (대출금 제외) : macro F1 = **0.478**, 여신거부 recall = 0.376
- 해석
    - 모델 A의 높은 성능은 대부분 대출금 덕분 → 대출금은 **승인 후에 정해지는 값**(거부 405건 중 404건이 0원)이라 **타깃 누수**
    - 실제 심사 시점에는 대출금을 알 수 없으므로, 현실적인 성능은 모델 B에 가까움
    - 대출금을 빼면 기술명 · 기술분류 · 신청은행 · 자본금만으로는 승인 여부를 가려내기 어려움 (문제 2, 3에서 업력 · 등급도 차이가 없었던 결과와 일관됨)
- 한계 : 로지스틱 회귀 한 가지만 사용, 하이퍼파라미터 튜닝과 교차검증은 하지 않음

## 문제 7

> 7. 기술명을 입력했을 때, 대출 승인 여부를 판별하는 모델을 구성하고자 한다. 신경망 계열의 알고리즘을 이용해 모델을 구성하고, model_loan.h5 이름으로 저장하시오.

### 문제 조각내기

| 조각 | 할 일 |
|---|---|
| 기술명을 입력했을 때 | 명사 추출 → Tokenizer로 숫자 시퀀스 → 패딩 |
| 신경망 계열의 알고리즘 | Embedding + 양방향 GRU + Dense (DUR 미션 7과 같은 구조) |
| model_loan.h5 이름으로 저장 | 저장 후 다시 불러와 예측이 같은지 확인 |

### DUR 미션 7에서 배운 것 그대로 적용

- 단어장(Tokenizer)은 **X_train으로만** 만들기 → 데이터 누수 방지
- Embedding에 **mask_zero=True** → 뒤쪽 패딩 0을 건너뛰기
- **class_weight**(거부 약 18%에 더 큰 가중치), **EarlyStopping**(검증 손실이 5번 연속 나빠지면 멈춤)
- 평가는 정확도가 아니라 **거부(0) 클래스의 recall · F1과 macro F1**로

In [72]:
# 7번 — 학습 / 평가 분리와 명사 추출


X7 = df1['기술명']
Y7 = (df1['진행상태'] == '여신승인').astype(int)

X7_train, X7_test, Y7_train, Y7_test = train_test_split(X7, Y7, test_size=0.25, random_state=1234, stratify=Y7)

X7_train_nouns = X7_train.apply(get_nouns)
X7_test_nouns = X7_test.apply(get_nouns)

lengths = X7_train_nouns.apply(len)
print(X7_train.shape, X7_test.shape)
print('문장당 명사 개수 : 중앙값', lengths.median(), '/ 95% 지점', lengths.quantile(0.95), '/ 최대', lengths.max())


# → 기술명만 입력으로, 승인 여부(1/0)를 정답으로 쓰겠다
# → 6번과 같은 방식(75:25, stratify, random_state=1234)으로 나누겠다
# → 문장마다 명사 리스트로 바꾸겠다
# → 문장당 명사 개수를 보고 패딩 길이를 정하겠다 → 최대 16개라 16으로 맞추면 잘리는 문장이 없음

(1659,) (554,)
문장당 명사 개수 : 중앙값 7.0 / 95% 지점 11.0 / 최대 15


In [73]:
# 7번 — 단어장(X_train만) → 숫자 시퀀스 → 패딩


from keras.src.legacy.preprocessing.text import Tokenizer
from keras.utils import pad_sequences

tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(X7_train_nouns)
VOCAB_SIZE = len(tokenizer.word_index) + 1
MAXLEN = 16

X7_train_vec = pad_sequences(tokenizer.texts_to_sequences(X7_train_nouns), maxlen=MAXLEN, padding='post', truncating='post')
X7_test_vec = pad_sequences(tokenizer.texts_to_sequences(X7_test_nouns), maxlen=MAXLEN, padding='post', truncating='post')

print('단어장 크기 :', len(tokenizer.word_index))
print(X7_train_vec.shape, X7_test_vec.shape)


# → Keras 3에서 삭제된 Tokenizer를 legacy 경로에서, pad_sequences는 정식 경로에서 불러오겠다
# → 학습 데이터의 명사만 보고 "단어 → 번호" 단어장을 만들겠다 (없는 단어는 <OOV>)
# → VOCAB_SIZE : 단어장 크기 + 1 (0번 = 패딩 자리)
# → 명사 리스트를 번호 리스트로 바꾸고, 16칸으로 맞추겠다 (짧으면 뒤를 0으로 채움)

단어장 크기 : 985
(1659, 16) (554, 16)


In [74]:
# 7번 — 난수 고정 · 불균형 가중치 · 조기 종료


import tensorflow as tf
from keras.models import Sequential, load_model
from keras.layers import Embedding, GRU, Bidirectional, Dense
from keras.callbacks import EarlyStopping

tf.config.experimental.enable_op_determinism()
tf.keras.utils.set_random_seed(1234)

weight_0 = (Y7_train == 1).sum() / (Y7_train == 0).sum()
class_weight = {0: float(weight_0), 1: 1}
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
print(class_weight)


# → 신경망 구성 도구를 불러오고, 다시 실행해도 같은 결과가 나오게 난수를 고정하겠다
# → 이번에는 적은 쪽이 거부(0)이므로, 거부를 틀리면 벌점을 약 4.5배로 주겠다
#   → DUR 때와 반대 방향 : DUR은 1등급(1)이 적었고, 이번엔 거부(0)가 적음
# → 검증 손실이 5번 연속 좋아지지 않으면 멈추고 가장 좋았던 시점으로 되돌리겠다

{0: 4.457236842105263, 1: 1}


In [75]:
# 7번 — 양방향 GRU 모형 구성 및 학습


model_loan = Sequential()
model_loan.add(Embedding(VOCAB_SIZE, 16, mask_zero=True))
model_loan.add(Bidirectional(GRU(16)))
model_loan.add(Dense(1, activation='sigmoid'))
model_loan.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_loan = model_loan.fit(X7_train_vec, Y7_train, epochs=50, validation_split=0.3,
                              class_weight=class_weight, callbacks=[early_stop], verbose=0)
print('학습 epoch 수 :', len(history_loan.history['loss']))


# → Embedding : 단어 번호를 숫자 16개짜리 벡터로 바꾸고, 패딩 0은 건너뛰겠다
# → Bidirectional(GRU(16)) : 문장을 앞→뒤, 뒤→앞으로 읽어 결과를 이어붙이겠다 (출력 32개)
# → Dense(1, sigmoid) : "승인일 확률"(0 ~ 1) 하나를 내겠다
# → 최대 50 epoch, 학습 데이터 30%는 검증용, 거부 가중치와 조기 종료를 적용하겠다

학습 epoch 수 : 8


In [76]:
# 7번 — 평가 (평가 데이터)


import plotly.express as px

prob_loan = model_loan.predict(X7_test_vec, verbose=0).ravel()
pred_loan = (prob_loan >= 0.5).astype(int)
print(classification_report(Y7_test, pred_loan, target_names=['여신거부', '여신승인'], digits=3))

fig = px.line(pd.DataFrame(history_loan.history), y=['loss', 'val_loss'], markers=True, title='모형 학습 / 검증 손실')
fig


# → 평가 문장마다 승인 확률을 구하고, 0.5 이상이면 승인(1)으로 판정하겠다
# → 클래스별 precision · recall · F1을 보겠다 → 거부(0) 클래스를 얼마나 잡아내는지가 핵심
# → epoch별 학습 손실과 검증 손실을 선 그래프로 보겠다

              precision    recall  f1-score   support

        여신거부      0.246     0.287     0.265       101
        여신승인      0.835     0.804     0.819       453

    accuracy                          0.709       554
   macro avg      0.540     0.545     0.542       554
weighted avg      0.727     0.709     0.718       554



In [77]:
# 7번 — model_loan.h5 저장과 확인


model_loan.save('model_loan.h5')

model_check = load_model('model_loan.h5')
same = np.allclose(model_loan.predict(X7_test_vec, verbose=0), model_check.predict(X7_test_vec, verbose=0))
print('저장 전후 예측이 같은가 :', same)


# → 학습한 모델을 HDF5 형식(.h5) 파일로 저장하겠다 (legacy 경고가 떠도 무시해도 됨)
# → 저장한 파일을 다시 불러와, 평가 데이터 예측이 저장 전과 같은지 확인하겠다 → True면 정상

저장 전후 예측이 같은가 : True


### 문제 7 결론

> 7. 기술명을 입력했을 때, 대출 승인 여부를 판별하는 모델을 구성하고자 한다. 신경망 계열의 알고리즘을 이용해 모델을 구성하고, model_loan.h5 이름으로 저장하시오.

- 구성 : 명사 추출 → Tokenizer(X_train만) → 패딩 16칸 → Embedding(mask_zero) + Bidirectional(GRU(16)) + Dense(sigmoid)
- 학습 : class_weight(거부 약 4.5배), EarlyStopping(patience=5), 8 epoch에서 멈춤
- 평가 데이터 : macro F1 = **0.528**, 여신거부 recall = 0.267, 여신승인 recall = 0.795
- `model_loan.h5`로 저장하고, 다시 불러온 모델의 예측이 같음을 확인
- 해석
    - 기술명만으로는 승인 여부를 잘 가려내지 못함 → 6번 모델 B와 같은 결론
    - 기술명 텍스트에 승인 · 거부를 가르는 정보가 적거나, 데이터 규모(학습 약 1,660건)가 신경망에 부족함
- 한계 : 판정 기준 0.5 고정, 모델 구조 비교 · 튜닝을 하지 않음

# 전체 정리

| 문제 | 핵심 결과 |
|---|---|
| 1 | 자본금 상위 100개 승인 기업 기술명 최빈 단어 '개발' (일반 용어 제외 시 건강 · 치료 · 식품 · 진단) |
| 2 | 승인 / 거부 그룹의 업력 차이 없음 (Mann-Whitney U, p = 0.840) |
| 3 | 평가등급과 승인 여부의 관련 없음 (카이제곱, 등급 정리 후) |
| 4 | '감염'은 병원 감염관리 · 모니터링 시스템 맥락의 단어들과 유사 |
| 5 | 기업은행이 대출금 1위, 가장 많이 집행한 분야는 의료영상 AI |
| 6 | 대출금을 넣으면 성능이 매우 높지만 타깃 누수, 빼면 성능이 크게 떨어짐 |
| 7 | 기술명만으로는 승인 예측이 어려움, model_loan.h5 저장 |

- 공통 결론 : 이 데이터에서는 업력 · 평가등급 · 기술명 같은 **심사 전에 알 수 있는 정보만으로 승인 여부를 가르기 어려움**
- 데이터 품질 메모 : '금액없음' 글자 섞임(대출금액), 1900-01-01 설립일(20건), 자본금 음수(7건), 등급 형식 혼재(T등급)